# Phase 1.9 — the colour-infrared re-encode (tile 32UNU, 115 cubes)

**This notebook computes nothing scientific.** It builds one artefact: a second
frozen-encoder cache in which the four 3-channel networks are fed
**(B8A, B04, B03)** — near-infrared in the red channel — instead of
**(B04, B03, B02)**.

**Why.** Every network encoder in this project is 3-channel, so all four were
denied **B8A**, the band where the vegetation signal mostly lives. The
hand-crafted `raw_features` baseline was not: it reads all four bands plus seven
NDVI statistics. P3 found `raw_features` is the *only* row separably above the
band-matched baseline — which leaves two very different readings
indistinguishable:

  **(a)** hand-crafted features beat learned representations, or
  **(b)** NIR beats RGB, and the representation is not what decides it.

This cache is how you tell them apart. Same weights, same extraction recipe,
same frame selection — **only the band routing changes**, so any difference
between a row and its `_cir` twin is band access and nothing else.

**Nothing published is overwritten.** The rgb cache at
`data/scaled_32UNU/embeddings/` is opened read-only; this writes to
`embeddings_cir/`, and the variant is part of the encoder *name*
(`dinov2_vitb14_cir`) so the two can never be confused on load.

Exit test at the bottom. **Step 10 is the load-bearing one.**

## Step 1: Install, then restart

In [ ]:
import importlib.util, os, IPython
SENTINEL = "/content/.phase1_9_installed"
try:
    import google.colab            # noqa: F401
    ON_COLAB = True
except ImportError:
    # find_spec("google.colab") is NOT equivalent: it raises rather than
    # returning None when the parent `google` package is absent.
    ON_COLAB = False

if not ON_COLAB:
    print("not on Colab: skipping the install and the restart.")
    print("Run against your own environment (pip install -r requirements.txt) "
          "and continue from Step 2.")
elif os.path.exists(SENTINEL):
    print("Already installed in this runtime, skipping.")
    print(f"(delete {SENTINEL} and re-run to force a reinstall)")
else:
    # Not -q. A pip resolution failure here is the likeliest cause of every
    # later failure, and -q hides it.
    # satlaspretrain-models is REQUIRED here and was not in Phase 1.5's list:
    # that phase read no embeddings, this one builds them.
    !pip install earthnet s3fs xarray zarr netCDF4 scikit-learn scipy satlaspretrain-models

    # Colab ships a CUDA-matched torch. Installing over it swaps in a CPU wheel
    # and makes every encoder far slower, so only act if something is MISSING.
    # This is the one notebook in the project where that distinction costs real
    # wall-clock: everything downstream is CPU-only by design.
    if (importlib.util.find_spec("torch") is None
            or importlib.util.find_spec("torchvision") is None):
        !pip install torch torchvision

    # Verify before restarting, so a broken install cannot reach the encoders.
    import subprocess, sys
    probe = ("import s3fs, xarray, zarr, netCDF4, earthnet, pandas, numpy, "
             "torch, torchvision, satlaspretrain_models, sklearn, scipy")
    r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout)
        print(r.stderr)
        raise RuntimeError(
            "Install did not take. Read the pip output above for the real "
            "conflict. Do not continue: Step 6 would fail to build the encoders."
        )

    open(SENTINEL, "w").write("ok")
    print("\n" + "=" * 70)
    print("INSTALL VERIFIED. RESTARTING THE RUNTIME NOW. This is expected.")
    print("When it comes back, continue from Step 2. Do not re-run this cell.")
    print("=" * 70)
    IPython.get_ipython().kernel.do_shutdown(True)

## Step 2: Bootstrap

The resolver block is character-identical to every phase notebook from 1.3 on and is pinned by `tests/test_notebook_resolver.py`.

In [ ]:
import os, sys, glob, zipfile, textwrap

REQUIRED = ["data/ndvi.py", "data/loader.py", "data/paths.py",
            "data/climatology.py", "encoders/manifest.py",
            "encoders/pipeline.py", "encoders/base.py", "encoders/frames.py",
            "encoders/raw_features.py", "encoders/imagenet_vit.py",
            "encoders/dinov2_vit.py", "encoders/satlas_s2.py",
            "encoders/satlas_s2_mi.py", "scripts/scale_p4.py",
            "probes/p3_forecast.py", "probes/p4_ceiling.py",
            "probes/cv.py", "probes/p1_appearance.py", "probes/p2_deltas.py",
            "tests/test_cv_folds.py", "tests/test_p2_deltas.py",
            "tests/conftest.py"]
ZIP_NAME = "phase1_9_repo.zip"
PHASE = "phase1_9"
INPUT_PHASE = "phase1_2"          # resolved by the shared block; NEVER read here

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
except ImportError:
    DRIVE = None
    print("not on Colab, assuming the repo is the current directory")

def looks_like_repo(d):
    return d and all(os.path.exists(os.path.join(d, f)) for f in REQUIRED)

REPO = None
if DRIVE:
    zips = glob.glob(f"{DRIVE}/**/{ZIP_NAME}", recursive=True)
    unzipped = [os.path.dirname(os.path.dirname(h))
                for d in ("*", "*/*", "*/*/*")
                for h in glob.glob(f"{DRIVE}/{d}/probes/cv.py")]
    unzipped = [d for d in unzipped if looks_like_repo(d)]

    if zips:
        REPO = os.path.dirname(zips[0])
        marker = os.path.join(REPO, "probes", "p3_forecast.py")
        # Re-extract when the zip is newer than what is on disk. Without this a
        # freshly uploaded zip is ignored because an old checkout sits next to
        # it, and you debug last week's code.
        stale = (not os.path.exists(marker)
                 or os.path.getmtime(zips[0]) > os.path.getmtime(marker))
        if stale:
            print(f"found {zips[0]}")
            print(f"extracting into {REPO} (zip is newer)")
            with zipfile.ZipFile(zips[0]) as zf:
                zf.extractall(REPO)
            print()
            print("=" * 70)
            print("THE NOTEBOOK FILE ON DISK WAS JUST REPLACED.")
            print("Colab is still showing the cells it opened. To pick up the")
            print("new ones: File > Open notebook > Google Drive, and open")
            print("   " + os.path.join(REPO, "notebooks"))
            print("Until you do, the .py files are new and these cells are old.")
            print("=" * 70)
        else:
            print(f"using existing checkout at {REPO} (zip is not newer)")
    elif unzipped:
        REPO = unzipped[0]
        print(f"found unzipped repo, no zip present: {REPO}")
else:
    # Off Colab, walk up from the working directory: running the notebook from
    # notebooks/ is normal and must not be mistaken for a missing checkout.
    d = os.getcwd()
    while not looks_like_repo(d) and os.path.dirname(d) != d:
        d = os.path.dirname(d)
    REPO = d

if not looks_like_repo(REPO):
    raise RuntimeError(textwrap.dedent(f"""
        Could not find the Phase 1.9 code.

        Fix, 2 minutes:
          1. Run make_zip.sh locally to build {ZIP_NAME}
          2. Open https://drive.google.com
          3. Make a NEW subfolder  My Drive / NeurIPS-CCAI-2026 / phase1_9
          4. Drag {ZIP_NAME} into it (do not unzip)
          5. Re-run this cell.

        One subfolder per phase is deliberate: deleting phase1_9/ removes
        everything Phase 1.9 created and nothing an earlier phase depends on.
        data/raw stays at the project root -- it is shared, not a phase.

        Searched under: {DRIVE}
        Needed all of: {REQUIRED}
        Resolved REPO = {REPO}
    """).strip())

os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")

from data.paths import RAW_DIR, describe_phase, phase_dir

# --- READ-ONLY inputs, resolved wherever they already live -----------------
# === RESOLVER (pinned by tests/test_notebook_resolver.py) -- BEGIN ===
# Extracted and exercised by that test against a simulated Drive tree, so
# the precedence rule below cannot silently regress into "first hit wins".
def _candidates(rel, pattern="*"):
    """Every directory on Drive that could be `rel`, with its file count.

    Searched: this checkout, then Drive one, two and three levels down. Three,
    because phases are subfolders of one project folder -- the Phase 1.2
    embeddings sit at
        MyDrive / NeurIPS-CCAI-2026 / phase1_2 / data/phase1_2/embeddings
    which is two wildcards, while the shared cubes at
        MyDrive / NeurIPS-CCAI-2026 / data/raw
    are one.
    """
    seen, out = set(), []
    cands = [os.path.join(REPO, rel)]
    if DRIVE:
        for depth in ("*", "*/*", "*/*/*"):
            cands += sorted(glob.glob(f"{DRIVE}/{depth}/{rel}"))
    for c in cands:
        c = os.path.abspath(c)
        if c in seen or not os.path.isdir(c):
            continue
        seen.add(c)
        out.append((c, len(glob.glob(os.path.join(c, pattern)))))
    return out


def _resolve(rel, pattern, label, foreign_phase=False):
    """Pick ONE directory, by evidence, and show every candidate considered.

    TAKING THE FIRST HIT IS NOT A SELECTION, and it cost a real run: a stale
    copy of data/phase1_2/embeddings sat INSIDE the phase1_3 checkout, the old
    "this checkout first" rule preferred it over the true Phase 1.2 folder, and
    the run died on a pre-schema file nobody knew was there.

    So: most files wins, and for ANOTHER phase's artefacts a directory inside
    THIS phase's checkout never beats one outside it, whatever the counts. That
    is the layout contract -- a phase reads its inputs in place and never owns
    a copy -- expressed as code rather than as a docstring.
    """
    cands = [(c, n) for c, n in _candidates(rel, pattern) if n > 0]
    if not cands:
        return os.path.join(REPO, rel), []        # the caller reports the gap
    repo_abs = os.path.abspath(REPO)

    def inside_repo(c):
        return os.path.commonpath([repo_abs, c]) == repo_abs

    # The penalty applies ONLY in a per-phase checkout. In a plain development
    # clone the repo root IS where data/phase1_2 belongs, so penalising "inside
    # the repo" there would be backwards -- and a warning that fires when
    # nothing is wrong is a warning nobody reads the second time.
    def demote(c):
        return foreign_phase and IS_PHASE_CHECKOUT and inside_repo(c)

    ranked = sorted(cands, key=lambda cn: (
        0 if demote(cn[0]) else -1,                           # outside first
        -cn[1],                                               # then the fullest
        len(cn[0]),                                           # then the shortest
    ))
    chosen = ranked[0][0]
    if len(cands) > 1:
        print(f"[resolve] {label}: {len(cands)} candidate directories hold files --")
        for c, n in ranked:
            mark = "  <- USING" if c == chosen else ""
            flag = "  [inside this checkout]" if inside_repo(c) else ""
            print(f"[resolve]     {n:>4} file(s)  {c}{flag}{mark}")
    if demote(chosen):
        print(f"[resolve] WARNING: {label} resolved INSIDE this phase's checkout:")
        print(f"[resolve]   {chosen}")
        print("[resolve] Another phase's artefacts do not belong here -- one phase")
        print("[resolve] reads another's in place and never owns a copy. This is")
        print("[resolve] almost certainly stale. Delete it and re-run Step 2 so")
        print("[resolve] the real directory is found.")
    return chosen, ranked


# Is this checkout a PHASE folder (Drive), or a plain clone (local dev)? The
# name settles it and covers both Drive layouts that have existed: the nested
# "NeurIPS-CCAI-2026/phase1_3" and the older sibling "…-2026-phase1_3".
IS_PHASE_CHECKOUT = PHASE in os.path.basename(os.path.abspath(REPO))

RAW, _raw_cands = _resolve(RAW_DIR, "*.nc", "RAW")
EMB_IN, _emb_cands = _resolve(os.path.join("data", INPUT_PHASE, "embeddings"),
                              "*.npz", "EMB_IN", foreign_phase=True)
os.makedirs(RAW, exist_ok=True)

# A phase checkout should not contain another phase's artefact tree at all,
# even an empty one: it shadows the real directory on every future run.
_intruder = os.path.join(REPO, "data", INPUT_PHASE)
if IS_PHASE_CHECKOUT and os.path.isdir(_intruder):
    print()
    print(f"[resolve] NOTE: {_intruder}")
    print(f"[resolve] exists inside the {PHASE} checkout. {INPUT_PHASE} "
          "artefacts belong in the")
    print(f"[resolve] {INPUT_PHASE} subfolder. Nothing here writes to it, but it "
          "will keep shadowing")
    print("[resolve] the real one until you delete it.")
# === RESOLVER -- END ===

# --- this phase's OWN outputs ----------------------------------------------
RESULTS = phase_dir(PHASE, "results")

n_cubes = len(glob.glob(os.path.join(RAW, "*.nc")))
n_emb = len(glob.glob(os.path.join(EMB_IN, "*.npz")))
print(f"\nREPO    {REPO}")
print(f"RAW     {RAW}   ({n_cubes} cubes)"
      + ("" if n_cubes else "   <- Step 4 downloads them"))
print(f"EMB_IN  {EMB_IN}   ({n_emb} .npz)")
print("        ^ the 20-cube Phase 1.2 cache. READ-ONLY here, and NEVER written")
print("          to: it is keyed to exactly 20 cubes and every published result")
print("          must stay reproducible from it. Step 10 reads it to prove the")
print("          scaled cache reproduces it on the cubes the two share.")
print(f"RESULTS {RESULTS}   (this phase writes here only)")
describe_phase(PHASE)

from data.ndvi import ndvi
from encoders import TIER_A
from encoders.manifest import build_manifest
from probes import cv
print(f"\nimports OK. canonical NDVI at {ndvi.__module__}, "
      f"splits at {cv.__name__}, modes {cv.MODES}")
print(f"encoders to cache ({len(TIER_A)}): {TIER_A}")
print("this notebook writes a CACHE, not a result. No probe runs here.")
for f in REQUIRED:
    print(f"  ok  {f}")


# --- shell helper, defined here so it can never be skipped ------------------
# Named sh(), not run(): IPython has a %run magic. If a helper called run() is
# ever undefined, automagic silently rewrites run("...") into %run("...") and
# reports a confusing error about a missing script instead of a NameError.
import shlex, subprocess

PY = shlex.quote(sys.executable)

def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")
    print(f"[exit 0] {cmd}")

print("helper ready: sh('<shell command>')")

## Step 3: Environment — this phase wants a GPU

Encoding is the only GPU-bound work in the project. Everything downstream is CPU-only by design.

In [ ]:
import glob, os, textwrap

import numpy as np, pandas as pd, sklearn, scipy
print(f"numpy {np.__version__} | pandas {pd.__version__} | "
      f"sklearn {sklearn.__version__} | scipy {scipy.__version__}")

import torch
print(f"torch {torch.__version__}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    DEVICE = "cuda"
    print(f"GPU  {p.name}, {p.total_memory / 1e9:.1f} GB  <- encoding will use this")
else:
    DEVICE = "cpu"
    print(textwrap.dedent("""
        No GPU. This still WORKS -- nothing here needs CUDA -- but 1580 frames
        through four networks will take a while. On Colab: Runtime > Change
        runtime type > T4 GPU, then re-run from Step 2.
    """).strip())
print(f"\nDEVICE = {DEVICE!r}")

# Python >= 3.10, and this one is not cosmetic. dinov2_vitb14 loads its code
# from torch.hub, and that code uses PEP 604 unions (`X | None`), so on 3.9 it
# dies with "unsupported operand type(s) for |: 'type' and 'NoneType'" -- a
# TypeError from inside a downloaded file, five steps after the real cause.
# Measured on this project's own 3.9 venv: the other four encoders build fine
# and only DINOv2 fails, which is exactly the kind of partial failure that
# produces a cache with a hole in it.
import sys
_py = sys.version_info
print(f"python {_py.major}.{_py.minor}.{_py.micro}")
assert _py >= (3, 10), (
    f"python {_py.major}.{_py.minor} is too old for dinov2_vitb14: its "
    "torch.hub code uses `X | None`, which is a syntax error before 3.10. The "
    "other four encoders would build and the cache would silently be missing "
    "one encoder for every cube. Colab is on 3.11+; if you are running this "
    "locally, use a 3.10+ interpreter."
)
print("\nNOTHING here is fine-tuned. Every encoder is frozen: .eval() and")
print("torch.no_grad(), re-asserted on every call by encoders.base.FrozenEncoder.")

## Step 4: The 115 cubes, and the two cache directories

`RGB_EMB` is the published cache and is **never written to**. `OUT_EMB` is new.

In [ ]:
TILE, N_CUBES = "32UNU", 115
_scaled_rel = os.path.join("data", f"scaled_{TILE}")

# Where the shared, non-phase artefacts live: beside data/raw, one level above a
# per-phase checkout. In a plain development clone that is the repo root.
PROJECT_ROOT = os.path.dirname(REPO) if IS_PHASE_CHECKOUT else REPO

CUBES, _cube_cands = _resolve(os.path.join(_scaled_rel, "raw"), "*.nc", "CUBES")
if not glob.glob(os.path.join(CUBES, "*.nc")):
    CUBES = os.path.join(PROJECT_ROOT, _scaled_rel, "raw")
    print(f"[scaled] nothing on disk yet; will download into {CUBES}")

SCALED_ROOT = os.path.dirname(CUBES)
RGB_EMB = os.path.join(SCALED_ROOT, "embeddings")   # READ-ONLY: the published RGB cache
OUT_EMB = os.path.join(SCALED_ROOT, "embeddings_cir")  # this notebook writes HERE
OUT_MSK = os.path.join(SCALED_ROOT, "masks")
os.makedirs(OUT_EMB, exist_ok=True)
os.makedirs(OUT_MSK, exist_ok=True)

from scripts.scale_p4 import download
CUBE_PATHS = download(TILE, "train", N_CUBES, CUBES)

n = len(CUBE_PATHS)
print(f"\nCUBES    {CUBES}   ({n} cubes)")
print(f"RGB_EMB  {RGB_EMB}   ({len(glob.glob(os.path.join(RGB_EMB, '*.npz')))} .npz, the PUBLISHED rgb cache -- READ-ONLY)")
print(f"OUT_EMB  {OUT_EMB}   (this notebook writes here, and ONLY here)")
print(f"OUT_MSK  {OUT_MSK}   (this notebook writes here)")
print(f"EMB_IN   {EMB_IN}   ({len(glob.glob(os.path.join(EMB_IN, '*.npz')))} .npz, "
      "the 20-cube Phase 1.2 cache -- READ-ONLY, for the Step 10 cross-check)")
assert n >= 20, f"only {n} cubes; the 64 px non-overlap rule caps 32UNU at 115"

# The 20 original cubes must be a strict subset, or Step 10 has nothing to check
# and no scaled result is comparable to a published one.
OLD_CUBES = {os.path.basename(p) for p in glob.glob(os.path.join(RAW, "*.nc"))}
NEW_CUBES = {os.path.basename(p) for p in CUBE_PATHS}
SHARED = sorted(OLD_CUBES & NEW_CUBES)
print(f"\n{len(OLD_CUBES)} original cubes, {len(NEW_CUBES)} scaled, "
      f"{len(SHARED)} shared, {len(NEW_CUBES - OLD_CUBES)} new")
assert OLD_CUBES <= NEW_CUBES, (
    f"{sorted(OLD_CUBES - NEW_CUBES)[:3]} are in data/raw but NOT in the scaled "
    "set. The 20-cube results would then not be a subset of the scaled ones and "
    "the two could not be compared.")

## Step 5: Unit tests

The invariant is **0 failed**, never a particular pass count.

In [ ]:
# pytest.ini already sets addopts = -q. Passing -q again makes it -qq,
# which hides the per-file progress.
sh(f"{PY} -m pytest tests")


## Step 6: Build the four CIR encoders

`build_encoder` strips the `_cir` suffix to find the wrapper and then re-asserts it against the instance's own `name`, so a variant can never be cached under the base name.

In [ ]:
import time
from encoders import TIER_A_CIR as TIER_A, build_encoder

print(f"TIER_A_CIR = {TIER_A}\n")
print("Same weights, same extraction recipe, same frame selection as the rgb\n"
      "run. The ONLY difference is which three bands reach the stem:\n"
      "(B8A, B04, B03) instead of (B04, B03, B02). raw_features has no _cir\n"
      "twin -- it already reads all four bands.\n")
t0 = time.time()
ENCODERS = {}
for name in TIER_A:
    ENCODERS[name] = build_encoder(name, device=DEVICE, verbose=True)
print(f"\n{len(ENCODERS)} encoders ready on {DEVICE} in {time.time() - t0:.0f}s")

for name, enc in ENCODERS.items():
    print(f"  {name:<24} D={enc.embed_dim:<5} grid_D={enc.grid_dim:<5} "
          f"window_len={enc.window_len}")
assert set(ENCODERS) == set(TIER_A)

## Step 7: Smoke test on ONE cube before committing to 460 files

In [ ]:
from data.loader import load_cube
from encoders.pipeline import encode_cube

probe = load_cube(sorted(CUBE_PATHS)[0], verbose=False)
print(f"smoke cube: {os.path.basename(probe.path)}  values {probe.values.shape}\n")

smoke, t0 = {}, time.time()
for name, enc in ENCODERS.items():
    ec = encode_cube(probe, enc, verbose=False)
    smoke[name] = ec
    assert ec.embeddings.ndim == 2 and ec.embeddings.shape[1] == enc.embed_dim
    assert ec.grid.shape[1:] == (16, enc.grid_dim), ec.grid.shape
    assert np.isfinite(ec.embeddings).all() and np.isfinite(ec.grid).all()
    print(f"  {name:<24} pooled {str(ec.embeddings.shape):<14} "
          f"grid {str(ec.grid.shape):<18} wsd max {ec.window_span_days.max():.0f} d")

# NOT smoke["raw_features"]: raw_features has no _cir twin (it already
# reads all four bands), so this roster is networks only. Any encoder
# serves as the reference -- the assertion below is that they all AGREE.
ts0 = next(iter(smoke.values())).timestamps
assert all(np.array_equal(ec.timestamps, ts0) for ec in smoke.values()), (
    "the encoders disagree about which frames were retained; frame selection "
    "happens BEFORE any network sees the data, so they cannot legitimately differ")
print(f"\nall five agree on {len(ts0)} retained frames | {time.time() - t0:.0f}s "
      f"for 1 cube x 5 encoders on {DEVICE}")
print(f"projected for {len(CUBE_PATHS)} cubes: "
      f"~{(time.time() - t0) * len(CUBE_PATHS) / 60:.0f} min")

## Step 8: The encode — resumable, because Colab sessions die

460 `.npz` = 115 cubes × 4 networks. The loop **skips whatever already exists**; a disconnected session is resumed by re-running this cell.

In [ ]:
from encoders.pipeline import (SCHEMA_VERSION, cube_masks, encode_cube,
                               load_encoded, save_encoded, save_masks)

print(f"cache schema v{SCHEMA_VERSION}. An older-schema file is REFUSED on load, "
      "never silently reused.")
print(f"writing to {OUT_EMB}\n")

rows, failures = [], []
t_start = time.time()
for i, path in enumerate(sorted(CUBE_PATHS), 1):
    cube = os.path.basename(path)
    stem = os.path.splitext(cube)[0]
    todo = [n for n in TIER_A
            if not os.path.exists(os.path.join(OUT_EMB, f"{stem}__{n}.npz"))]
    mask_path = os.path.join(OUT_MSK, f"{stem}__masks.npz")
    if not todo and os.path.exists(mask_path):
        rows += [{"cube": cube, "encoder": n, "status": "cached"} for n in TIER_A]
        continue

    try:
        s = load_cube(path, verbose=False)
    except Exception as e:                      # noqa: BLE001 -- reported, not hidden
        failures.append((cube, "load_cube", f"{type(e).__name__}: {e}"))
        print(f"[{i:>3}/{len(CUBE_PATHS)}] FAILED to load {cube}: {e}")
        continue

    if not os.path.exists(mask_path):
        save_masks(OUT_MSK, cube_masks(s, verbose=False), verbose=False)

    t0 = time.time()
    for name in TIER_A:
        out = os.path.join(OUT_EMB, f"{stem}__{name}.npz")
        try:
            if os.path.exists(out):
                ec, status = load_encoded(out), "cached"
            else:
                ec = encode_cube(s, ENCODERS[name], verbose=False)
                save_encoded(OUT_EMB, ec, verbose=False)
                status = "encoded"
            rows.append({"cube": cube, "encoder": name, "status": status,
                         "T_kept": int(ec.embeddings.shape[0]),
                         "D": int(ec.embeddings.shape[1])})
        except Exception as e:                  # noqa: BLE001
            failures.append((cube, name, f"{type(e).__name__}: {e}"))
            print(f"[{i:>3}/{len(CUBE_PATHS)}] FAILED {name} on {cube}: {e}")

    done = sum(1 for r in rows if r["status"] == "encoded")
    el = time.time() - t_start
    print(f"[{i:>3}/{len(CUBE_PATHS)}] {cube[:52]:<52} "
          f"{time.time() - t0:5.1f}s | {done:>4} encoded | "
          f"elapsed {el / 60:5.1f} min | eta {el / i * (len(CUBE_PATHS) - i) / 60:5.1f} min",
          flush=True)

ENCODE_LOG = pd.DataFrame(rows)
print(f"\n{len(ENCODE_LOG)} (cube, encoder) pairs in {(time.time() - t_start) / 60:.1f} min")
print(ENCODE_LOG.status.value_counts().to_string())
if failures:
    print(f"\n{len(failures)} FAILURES:")
    for f in failures[:20]:
        print("  ", f)

## Step 9: Audit the cache — all 115 × 4, or it is not usable

In [ ]:
from encoders.pipeline import (assert_embeddings_complete, audit_embeddings,
                               print_embedding_audit)

CUBE_IDS = {os.path.basename(p) for p in CUBE_PATHS}
AUDIT = audit_embeddings(OUT_EMB, cube_ids=CUBE_IDS)
print()
assert_embeddings_complete(AUDIT, CUBE_IDS, TIER_A)

n_emb = len(glob.glob(os.path.join(OUT_EMB, "*.npz")))
n_msk = len(glob.glob(os.path.join(OUT_MSK, "*.npz")))
print(f"\nembeddings {n_emb} .npz  (expected {len(CUBE_IDS)} x {len(TIER_A)} "
      f"= {len(CUBE_IDS) * len(TIER_A)})")
print(f"masks      {n_msk} .npz  (expected {len(CUBE_IDS)})")
assert n_emb == len(CUBE_IDS) * len(TIER_A), "the embedding cache has holes"
assert n_msk == len(CUBE_IDS), "the mask cache has holes -- common-masking needs it"
mb = sum(os.path.getsize(p) for p in glob.glob(os.path.join(OUT_EMB, "*.npz")))
print(f"cache size {mb / 1e6:.0f} MB")

## Step 10: THE CROSS-CHECK — same frames, different pixels

This is the cell that makes the whole phase trustworthy, and it asserts in
**two directions**:

- **Frame selection, timestamps, clear fractions: BIT-IDENTICAL** to the rgb
  cache. These come from the cube and the mask rule, *before* any network sees
  anything, so a single difference means the two caches are not the same
  experiment and no rgb-vs-cir comparison is valid.
- **Embeddings: MATERIALLY DIFFERENT.** If they matched, the band routing
  silently did nothing and the whole phase is a no-op that would look like a
  clean null result.

In [ ]:
from encoders.pipeline import load_encoded
from encoders import TIER_A_CIR

rows, n_same = [], 0
for cube in sorted({os.path.basename(p) for p in CUBE_PATHS}):
    stem = os.path.splitext(cube)[0]
    for cir in TIER_A_CIR:
        base = cir[: -len("_cir")]
        p_rgb = os.path.join(RGB_EMB, f"{stem}__{base}.npz")
        p_cir = os.path.join(OUT_EMB, f"{stem}__{cir}.npz")
        if not os.path.exists(p_rgb):
            continue
        a, b = load_encoded(p_rgb), load_encoded(p_cir)
        # --- must be IDENTICAL: everything decided before the network ---
        np.testing.assert_array_equal(a.kept_idx, b.kept_idx,
            err_msg=f"{cube}/{base}: frame selection differs between caches")
        np.testing.assert_array_equal(np.asarray(a.timestamps), np.asarray(b.timestamps),
            err_msg=f"{cube}/{base}: timestamps differ")
        np.testing.assert_allclose(a.clear_frac, b.clear_frac, rtol=0, atol=0,
            err_msg=f"{cube}/{base}: clear fractions differ")
        assert a.embeddings.shape == b.embeddings.shape, (cube, base)
        # --- must DIFFER: the band routing has to have done something ---
        d = float(np.abs(a.embeddings - b.embeddings).max())
        rel = d / max(float(np.abs(a.embeddings).max()), 1e-12)
        n_same += int(d == 0.0)
        rows.append({"cube": cube, "encoder": base, "max_abs_diff": d, "rel": rel})

CIR_CHECK = pd.DataFrame(rows)
print(f"{len(CIR_CHECK)} (cube, encoder) pairs cross-checked against the rgb cache")
print("\nframe selection / timestamps / clear_frac : BIT-IDENTICAL on all pairs")
print(f"embeddings identical on {n_same} pairs (must be 0)\n")
print(CIR_CHECK.groupby("encoder").max_abs_diff.agg(["min", "median", "max"]).to_string())
assert n_same == 0, (
    f"{n_same} (cube, encoder) pairs produced BYTE-IDENTICAL embeddings under "
    "two different band composites. The band routing did nothing -- every _cir "
    "row would be a relabelled copy of its rgb twin, and the resulting 'no "
    "difference from NIR' would be an artefact of this notebook, not a finding."
)
assert (CIR_CHECK.rel > 1e-3).all(), (
    "some pair differs only at float noise; check composite_from_s2 is actually "
    "reaching that wrapper's forward pass"
)
print("\nPASS: same frames, materially different pixels. The two caches are the "
      "same experiment with one variable changed.")


## Step 11: Report

In [ ]:
RESULTS = phase_dir(PHASE, "results")
ENCODE_LOG.to_csv(os.path.join(RESULTS, "phase1_9_cir_cache.csv"), index=False)
CIR_CHECK.to_csv(os.path.join(RESULTS, "phase1_9_cir_vs_rgb_check.csv"), index=False)
print(f"wrote {RESULTS}/phase1_9_cir_cache.csv")
print(f"      {RESULTS}/phase1_9_cir_vs_rgb_check.csv")
n_emb = len(glob.glob(os.path.join(OUT_EMB, "*.npz")))
print(f"\n{n_emb} .npz at {OUT_EMB}")
print("\nDOWNLOAD embeddings_cir/ back to data/scaled_32UNU/embeddings_cir/ ,")
print("then run P3 locally with the 9 encoder views (5 rgb + 4 cir).")
describe_phase(PHASE)


## Phase 1.9 is done when

- [ ] Step 5 reports **0 failed**.
- [ ] Step 8 reports **0 failures** and 460 `(cube, encoder)` pairs.
- [ ] Step 9's audit passes: 460 `.npz`, no holes.
- [ ] **Step 10 passes both directions** — frame selection bit-identical to the
      rgb cache, embeddings materially different on every pair. Without this the
      cir cache is either a different experiment or a relabelled copy, and
      either way the NIR comparison is void.
- [ ] `embeddings_cir/` downloaded to `data/scaled_32UNU/embeddings_cir/`.

Then: Tier-1 items 2 and 4 locally, and the combined P3 re-run.